In [11]:
import ollama
import pandas as pd
import json

In [12]:
emails = [
    {
        "id": 1,
        "subject": "AI Internship Application",
        "body": """
        Hello HR Team,

        I am a Computer Science student with experience in Python,
        Machine Learning, and AI projects. I would like to apply
        for your AI internship program.

        Best regards,
        Ahmed
        """
    },

    {
        "id": 2,
        "subject": "Question About Internship",
        "body": """
        Hello,

        I would like to know when the internship applications
        will open.

        Thank you.
        """
    },

    {
        "id": 3,
        "subject": "AI Engineer Application",
        "body": """
        Dear HR Team,

        I have 2 years of experience in Python, Machine Learning,
        and AI automation. I am interested in joining your company
        as an AI Engineer.

        Best regards,
        Mohamed
        """
    },

    {
        "id": 4,
        "subject": "General Question",
        "body": """
        Hello,

        Do you offer training programs for students?

        Thanks.
        """
    }
]

In [13]:
def classify_email(email):

    prompt = f"""
You are an HR assistant.

Analyze the following student email.

Classify the email based on its importance to the HR department.

Priority rules:

High:
- The student is applying for a job or internship.
- The student has relevant AI, Python, Machine Learning,
  or technical experience.

Medium:
- The student is interested in an opportunity.
- The email may be relevant but does not contain enough
  information about experience.

Low:
- The email is only a general question.
- The email is not a direct application.

Return ONLY valid JSON in this format:

{{
    "category": "Job Application / Internship Application / General Question",
    "priority": "High / Medium / Low",
    "reason": "Short explanation"
}}

Email Subject:
{email['subject']}

Email Body:
{email['body']}
"""

    response = ollama.chat(
        model="phi3:3.8b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    result = response["message"]["content"]

    return result

In [16]:
results = []

for email in emails:

    ai_result = classify_email(email)

    parsed_result = parse_result(ai_result)

    results.append({
        "id": email["id"],
        "subject": email["subject"],
        "category": parsed_result["category"],
        "priority": parsed_result["priority"],
        "reason": parsed_result["reason"]
    })

In [17]:
def parse_result(result):

    try:
        return json.loads(result)

    except json.JSONDecodeError:
        return {
            "category": "Unknown",
            "priority": "Low",
            "reason": "Could not parse AI response"
        }

In [18]:
df = pd.DataFrame(results)

df

,id,subject,category,priority,reason
0,1,AI Internship Application,Unknown,Low,Could not parse AI response
1,2,Question About Internship,Unknown,Low,Could not parse AI response
2,3,AI Engineer Application,Job Application,High,The student is applying for a job as an AI Eng...
3,4,General Question,Unknown,Low,Could not parse AI response


In [19]:
priority_order = {
    "High": 1,
    "Medium": 2,
    "Low": 3
}

df["priority_score"] = df["priority"].map(priority_order)

df = df.sort_values("priority_score")

df

,id,subject,category,priority,reason,priority_score
2,3,AI Engineer Application,Job Application,High,The student is applying for a job as an AI Eng...,1
0,1,AI Internship Application,Unknown,Low,Could not parse AI response,3
1,2,Question About Internship,Unknown,Low,Could not parse AI response,3
3,4,General Question,Unknown,Low,Could not parse AI response,3


In [20]:
print("===== HR PRIORITY EMAIL LIST =====\n")

for _, row in df.iterrows():

    print(f"Subject: {row['subject']}")
    print(f"Priority: {row['priority']}")
    print(f"Category: {row['category']}")
    print(f"Reason: {row['reason']}")
    print("-" * 50)

===== HR PRIORITY EMAIL LIST =====

Subject: AI Engineer Application
Priority: High
Category: Job Application
Reason: The student is applying for a job as an AI Engineer with relevant experience in Python, Machine Learning, and AI automation.
--------------------------------------------------
Subject: AI Internship Application
Priority: Low
Category: Unknown
Reason: Could not parse AI response
--------------------------------------------------
Subject: Question About Internship
Priority: Low
Category: Unknown
Reason: Could not parse AI response
--------------------------------------------------
Subject: General Question
Priority: Low
Category: Unknown
Reason: Could not parse AI response
--------------------------------------------------
